Environment: harpy

## Import packages

In [ ]:
import sparrow as sp
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import os

from sparrow.table import add_table_layer
from sparrow.shape import add_shapes_layer
from spatialdata.models import ShapesModel
from spatialdata.models import TableModel
from sparrow.utils._keys import _INSTANCE_KEY, _REGION_KEY
import geopandas as gpd
import dask.array as da
import numpy as np
from skimage import io, img_as_ubyte

## Specify paths

In [2]:
INPUT_DIR = 'D:/Data/2024-07-UTBenchmark-SpC/data/processed/BMK/exp74/05.AllheStat/BSTViewer_project'
OUTPUT_DIR = 'D:/Data/2024-07-UTBenchmark-SpC/data/processed/BMK/exp74'

## Create sdata object and add images

In [ ]:
img_path = os.path.join(INPUT_DIR, 'he.tif')

sdata = sp.io.create_sdata(
    input = img_path, 
    output_path = os.path.join(OUTPUT_DIR, 'sdata.zarr'), 
    img_layer = 'raw_image', 
    chunks = 1024)

## Add BMK data to sdata

In [ ]:
# Get all available levels (= all available folders)
directory = os.path.join(INPUT_DIR, 'subdata')
level_names = [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]
level_paths = {name: os.path.join(directory, name) for name in level_names}
print(level_names)

In [4]:
# Define function to add BMK level to sdata
def add_BMK_level(sdata, level_name, level_path, hires_img):
    def calculate_scaling_factor(width, height):
        std_width = 1000
        std_height = std_width / (46 * 31) * (46 * 36 * np.sqrt(3) / 2.0)
        
        if std_width / std_height > width / height:
            scale = width / std_width
        else:
            scale = height / std_height
            
        return scale
    
    def calculate_radius(coordinates, subset_size=1000):
        x_center = hires_img.shape[2]/2
        y_center = hires_img.shape[1]/2

        x_min = x_center - (subset_size / 2)
        x_max = x_center + (subset_size / 2)
        y_min = y_center - (subset_size / 2)
        y_max = y_center + (subset_size / 2)

        subset_coords = coordinates[
            (coordinates[:, 0] >= x_min) & (coordinates[:, 0] <= x_max) &
            (coordinates[:, 1] >= y_min) & (coordinates[:, 1] <= y_max)
        ]
        
        distances = np.linalg.norm(subset_coords[:, np.newaxis] - subset_coords[np.newaxis, :], axis=-1)
        
        np.fill_diagonal(distances, np.inf) 
        min_distance = np.min(distances)

        radius = (min_distance / 2) * 0.98
        return radius

    # Calculate scaling factor
    scaling_factor = calculate_scaling_factor(hires_img.shape[2], hires_img.shape[1])

    # Read in expression data and coordinates as adata
    adata = sc.read_10x_mtx(level_path, var_names='gene_symbols', cache=True)
    barcode_pos = pd.read_csv(f'{level_path}/barcodes_pos.tsv.gz', compression='gzip', sep='\t', names=["barcode","array_col","array_row"], header=None)
    coordinates = barcode_pos[['array_col','array_row']].to_numpy() * scaling_factor
    adata.obsm["spatial"] = coordinates

    # Add region key and change indexes
    adata.obs[_REGION_KEY] = f"labels_spots_{level_name}"
    adata.obs['cell_ID'] = [int(x.split('_')[1]) for x in adata.obs.index]
    adata.obs.index = adata.obs['cell_ID']
    adata.obs.index.name = 'cells'

    # Add shapes and labels layer based on coordinates
    radius = calculate_radius(coordinates)
    radii = np.full(len(adata.obsm['spatial']), radius)
    spots = ShapesModel.parse(coordinates, geometry=0, radius=radii)
    spots.index = adata.obs.index
    polygons = spots.buffer(radius, cap_style="round")
    
    sdata = add_shapes_layer(sdata, gpd.GeoDataFrame(geometry=polygons), output_layer=f'spots_{level_name}', overwrite=True)
    sdata = sp.im.add_labels_layer_from_shapes_layer(sdata, shapes_layer=f"spots_{level_name}", output_layer=f"labels_spots_{level_name}", overwrite=True)

    # Add adata as table layer
    sdata = add_table_layer(sdata, adata=adata.copy(), output_layer=f'table_{level_name}', region=[f"labels_spots_{level_name}"], overwrite=True)

    return sdata


In [ ]:
# Run function for all levels
hires_img = sdata['raw_image'].compute().data

for level_name, level_path in level_paths.items():
    print(f'running: {level_name}')
    
    sdata = add_BMK_level(
        sdata,
        level_name = level_name,
        level_path = level_path,
        hires_img = hires_img,
    )

In [ ]:
sp.pl.plot_shapes(
    sdata,
    img_layer = "raw_image",
    shapes_layer="spots_L13_heAuto",
    # crd = [7000,8000,7000,8000],
    channel=0,
    table_layer='table_L13_heAuto',
    column='Plp1',
    alpha=1,
    linewidth=0,
    figsize = (25,25)
)

## Save raw Anndata objects

In [9]:
for level_name in level_names:
    # Save adata with H&E added
    adata = sdata.tables[f'table_{level_name}'].copy()

    library_id = level_name
    adata.uns["spatial"] = {library_id: {}}
    adata.uns["spatial"][library_id]["images"] = {}
    adata.uns['spatial'][library_id]['images']['hires'] = img_as_ubyte(hires_img.transpose(1, 2, 0))
    adata.uns['spatial'][library_id]['use_quality'] = 'hires' 
    adata.uns['spatial'][library_id]['scalefactors'] = {"tissue_hires_scalef": 1}

    adata.write(f'{OUTPUT_DIR}/adata_{level_name}.h5ad')

## Leiden clustering

In [ ]:
import gc # garbage collection to avoid out-of-memory issues.

for level_name in level_names:
    if level_name not in ['L1_heAuto', 'L2_heAuto']:
        print(f'running: {level_name}')
        
        # Preprocess data
        sdata = sp.tb.preprocess_transcriptomics(
            sdata=sdata,
            size_norm=False,
            labels_layer=f"labels_spots_{level_name}",
            table_layer=f"table_{level_name}",
            output_layer=f"table_{level_name}_processed",
            min_counts=50,
            min_cells=5,
            highly_variable_genes=True,
            max_value_scale=10,
            n_comps = 50,
            update_shapes_layers=False,
            overwrite=True
        )
        
        gc.collect()
        
        # Leiden clustering
        leiden_resolutions = [0.2, 0.4, 0.6]

        for res in leiden_resolutions:
            res_str = str(res).replace('.', '')
            
            sdata = sp.tb.leiden(
                sdata, 
                labels_layer=f"labels_spots_{level_name}",
                table_layer=f"table_{level_name}_processed",
                output_layer=f"table_{level_name}_processed",
                calculate_umap=True,
                calculate_neighbors=True,
                rank_genes=True,
                n_neighbors=35,
                n_pcs=30,
                resolution=res,
                key_added=f'leiden_{res_str}',
                index_names_var=None,
                index_positions_var=None,
                random_state=100,
                overwrite=True
            )
        
        gc.collect()
        
        # Save adata with H&E added
        adata = sdata.tables[f'table_{level_name}_processed'].copy()

        library_id = level_name
        adata.uns["spatial"] = {library_id: {}}
        adata.uns["spatial"][library_id]["images"] = {}
        adata.uns['spatial'][library_id]['images']['hires'] = img_as_ubyte(hires_img.transpose(1, 2, 0))
        adata.uns['spatial'][library_id]['use_quality'] = 'hires' 
        adata.uns['spatial'][library_id]['scalefactors'] = {"tissue_hires_scalef": 1}

        adata.write(f'{OUTPUT_DIR}/adata_{level_name}_processed.h5ad')